<a href="https://colab.research.google.com/github/stewari23/ITCS-3162/blob/main/lab_02_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Exercise: AI Tools & the Data Mining Pipeline

**ITCS 3162 — Introduction to Data Mining**

**Name:** Shivam Tewari
**Date:** 5-24-26

Complete the exercises below. **Some questions ask you to use an AI assistant** — when you do, you must:
1. Paste the prompt you used in the markdown cell labeled "PROMPT".
2. Paste the AI's response (or key parts) in the cell labeled "AI RESPONSE".
3. Write your own evaluation in "YOUR EVALUATION".

This is the appropriate-use pattern: prompt, response, your judgment.


## Exercise 1 — Identify appropriate use (15 pts)

For each scenario, decide whether the AI use is **appropriate** or **inappropriate** and explain why in one sentence.

| # | Scenario |
|---|---|
| A | A student doesn't understand what "stratified sampling" means and asks an AI to explain it with an example. |
| B | A student pastes the full text of an exercise question and submits the AI's answer as their reflection. |
| C | A student has a `KeyError` they can't figure out and asks the AI what it means. |
| D | A student asks the AI to write all their markdown answers in this notebook so they can focus on the code. |
| E | A student writes their own code, then asks the AI to suggest more idiomatic alternatives, then chooses one and explains it in their own words. |

YOUR ANSWERS:

- **A:** Appropriate because AI is being used to learn something.
- **B:** Inappropriate because AI is being used as a shortcut.
- **C:** Appropriate because the student first tried to figure out what the error means on their own.
- **D:** Inappropriate because AI isn't being used to learn how to write markdown.
- **E:** Appropriate because the student first did their own work and is now using AI to assist.


## Exercise 2 — Rewrite a weak prompt (35 pts)

Below is a weak prompt. Rewrite it so it's specific, scoped, and verifiable. Imagine you have a DataFrame `df` with a column `price` (numeric) and you want to detect rows whose price is unusually high.

**Weak prompt:** "find weird stuff"

YOUR REWRITTEN PROMPT:



## Exercise 3 — Prompt, response, evaluate (20 pts)

Use any AI assistant (ChatGPT, Claude, Copilot chat, Gemini, etc.) to answer this question:

> "In scikit-learn, what is the difference between `fit`, `transform`, and `fit_transform`? When would I use each?"

Fill in all three cells below.


**PROMPT** (what you actually typed):
In scikit-learn, what is the difference between fit, transform, and fit_transform? When would I use each?

Use research and evidence to create your response


**AI RESPONSE** (paste it here, or summarize the key points):

In scikit-learn, these methods are part of a consistent estimator/transformer API:

| Method                     | What it does                                | When to use it                                           |
| -------------------------- | ------------------------------------------- | -------------------------------------------------------- |
| `fit(X, y=None)`           | Learns parameters from data.                | Use on **training data** to learn what the object needs. |
| `transform(X)`             | Applies already-learned parameters to data. | Use on **validation/test/new data** after fitting.       |
| `fit_transform(X, y=None)` | Fits, then transforms, usually in one step. | Use on **training data** for transformers.               |

## `fit`: learn from the data

`fit` estimates whatever internal values the object needs. For example, `StandardScaler.fit(X_train)` learns the mean and standard deviation of each feature. scikit-learn documents `StandardScaler` as standardizing using `z = (x - u) / s`, where `u` and `s` are computed from the **training samples**. ([Scikit-learn][1])

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)  # learns mean and standard deviation from X_train
```

For a model like logistic regression, `fit` means “train the model.” For a transformer like a scaler, encoder, imputer, or PCA, `fit` means “learn the transformation rules.”

## `transform`: apply what was learned

`transform` uses the parameters learned during `fit` and applies them to data. It does **not** relearn anything.

```python
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

This is crucial for test data. You should not fit preprocessing on the test set, because that lets information from the test set influence your pipeline. Instead, learn preprocessing parameters from the training data, then apply the same transformation to test or production data.

Example:

```python
scaler = StandardScaler()

scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

Here, `X_test` is scaled using the training-set mean and standard deviation, not its own.

## `fit_transform`: fit and transform in one call

`fit_transform` is shorthand for:

```python
transformer.fit(X_train)
X_train_transformed = transformer.transform(X_train)
```

So this:

```python
X_train_scaled = scaler.fit_transform(X_train)
```

is usually equivalent to:

```python
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
```

scikit-learn maintainers note that `fit_transform` generally calls `fit` and then `transform`, unless an estimator overrides it with a faster specialized implementation. ([GitHub][2])

## Typical usage pattern

```python
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
model = LogisticRegression()

# Training phase
X_train_scaled = scaler.fit_transform(X_train)
model.fit(X_train_scaled, y_train)

# Test / inference phase
X_test_scaled = scaler.transform(X_test)
predictions = model.predict(X_test_scaled)
```

Use `fit_transform` on the training data, then use `transform` on all later data.

## In pipelines

Pipelines automate this pattern. scikit-learn’s pipeline documentation says that a pipeline applies `transform` through intermediate steps, then calls the final estimator’s prediction/scoring method. ([Scikit-learn][3]) The user guide also explains that if the final step has `predict`, the pipeline transforms the data through earlier steps before prediction. ([Scikit-learn][4])

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

pipe.fit(X_train, y_train)      # scaler.fit_transform(X_train), then model.fit(...)
pipe.predict(X_test)            # scaler.transform(X_test), then model.predict(...)
```

This is the safest and cleanest approach because it reduces the chance of data leakage.

## Rule of thumb

Use:

```python
fit
```

when you need to **learn from training data**.

Use:

```python
transform
```

when you need to **apply a learned transformation** to validation, test, or new data.

Use:

```python
fit_transform
```

when you are working with **training data** and want to learn the transformation and apply it immediately.

The most common mistake is doing this:

```python
X_test_scaled = scaler.fit_transform(X_test)  # usually wrong
```

That refits the scaler on the test data. The correct version is:

```python
X_test_scaled = scaler.transform(X_test)
```

[1]: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html?utm_source=chatgpt.com "StandardScaler — scikit-learn 1.8.0 documentation"
[2]: https://github.com/scikit-learn/scikit-learn/discussions/24620?utm_source=chatgpt.com "fit and fit_transform implementation details #24620"
[3]: https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html?utm_source=chatgpt.com "Pipeline — scikit-learn 1.8.0 documentation"
[4]: https://scikit-learn.org/stable/modules/compose.html?utm_source=chatgpt.com "7.1. Pipelines and composite estimators"


**YOUR EVALUATION** — answer all three:
1. Was the response correct?
2. Was anything missing or unclear?
3. In your own words (no copy-paste), explain the difference between `fit` and `transform`.

YOUR ANSWER:

1. How did you verify? The response was correct and it was verified by checking the sources that GPT used.
2. The response was clear.
3. In scikit-learn, fit means “learn from the data,” transform means “use what was learned to change the data,” and fit_transform does both in one step. You use fit when you want a model or tool to learn patterns from training data, transform when you want to apply that same learned change to new data.

## Exercise 4 — Map the pipeline (15 pts)

Below is a list of activities. For each, write which **pipeline stage** it belongs to (Problem understanding / Load / Explore / Preprocess / Model / Evaluate).

| # | Activity | Stage |
|---|---|---|
| A | Running `df.describe()` to see column means and quartiles |Explore|
| B | Choosing whether to use a decision tree or logistic regression |Model|
| C | Replacing missing ages with the median age |Preprocess|
| D | Talking to a stakeholder to clarify what "customer churn" means |Problem understanding|
| E | Computing accuracy and confusion matrix on the held-out test set |Evaluate|
| F | Reading a CSV with `pd.read_csv("data.csv")` |Load|
| G | One-hot encoding a `country` column |Preprocess|
| H | Plotting a histogram of `salary` |Explore|


In [ ]:
# Exercise 5 — Build your own mini-pipeline (20 pts)
#
# Using the wine dataset (built into sklearn), build a complete pipeline that:
#   1. Loads the data as a DataFrame
#   2. Prints the shape and the class distribution of the target
#   3. Splits into 70/30 train/test with random_state=42, stratified on the target
#   4. Builds a sklearn Pipeline with StandardScaler + LogisticRegression(max_iter=500)
#   5. Fits on the training data and reports test accuracy
#
# Fill in the TODOs below.

import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. LOAD
wine = load_wine(as_frame=True)
df = wine.frame

print("Shape:", df.shape)
print("Class distribution:")
print(df["target"].value_counts())

# 2. SPLIT
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# 3. PIPELINE
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=500))
])

# 4. FIT + SCORE
pipe.fit(X_train, y_train)

accuracy = pipe.score(X_test, y_test)
print(f"Test accuracy: {accuracy:.3f}")


## Exercise 6 — Reflection (15 pts)

In 4–6 sentences, answer **both**:

1. What's one task in this course you think AI tools would help you with the most, and one task where you'd intentionally not use them? Why?
2. Looking at the six-stage pipeline, which stage do you expect to find hardest and why?

YOUR ANSWER:

AI tools would help me most when I get stuck writing or checking code, because they can act like a patient tutor who points out small mistakes and explains them in simple words. I would intentionally not use AI tools when I need to understand the main idea of an assignment for myself, because if AI does all the thinking, I might copy an answer without really learning it. In the six-stage pipeline, I expect Preprocess to be the hardest stage because it is like cleaning and organizing a messy room before you can use it. You have to fix missing values, change words into numbers, and make the data ready for the model. That seems harder than just loading the data or checking accuracy because small cleaning choices can affect everything that happens later.

## Submission checklist

- [ ] Name and date filled in at the top
- [ ] All `YOUR ANSWER` / `TODO` prompts completed
- [ ] Exercise 3 includes a PROMPT, AI RESPONSE, and YOUR EVALUATION
- [ ] Code cell in Exercise 5 runs and prints test accuracy
- [ ] **Restart & Run All** completes without errors
- [ ] Downloaded as `.ipynb` and submitted via Canvas
